# FrozenLake UCBVI

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import sys

# Add project root to path for imports
script_path = Path().resolve()
project_root = script_path.parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import our experiment functions
from test_frozenlake_ucbvi import (
    run_ucbvi_experiment, 
    plot_results, 
    replay_episode
)

/opt/homebrew/anaconda3/envs/pmml/lib/python3.11/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment causal_gym/WindyGridWorld-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/opt/homebrew/anaconda3/envs/pmml/lib/python3.11/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment causal_gym/CartPoleWind-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/opt/homebrew/anaconda3/envs/pmml/lib/python3.11/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment Custom-LavaCrossing-easy-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/opt/homebrew/anaconda3/envs/pmml/lib/python3.11/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment Custom-LavaCrossing-hard-v0 already in registry.
  logger.war

## Configure Experiment Parameters
Let's set up our experiment with the following configuration:
- 500 episodes
- 25 steps per episode horizon
- Slippery and windy environment
- Epsilon-greedy exploration

In [2]:
# Experiment Parameters
NUM_EPISODES = 500
EPISODE_HORIZON = 25
PLANNING_SWEEPS = EPISODE_HORIZON
DELTA = 0.01
AGENT_EPSILON = 0.3  # Higher epsilon for more exploration

# Environment Configuration
ENV_CONFIG = {
    "map_name": "4x4",
    "is_slippery": True,  # Enable slippery mode
    "wind_probabilities": (0.5, 0.125, 0.125, 0.125, 0.125),  # Wind enabled
    "render_mode": "rgb_array",  # For replay capture
    "seed": 42  # Fixed seed for reproducibility
}

print("Starting UCBVI experiment with:")
print(f"Episodes: {NUM_EPISODES}")
print(f"Horizon: {EPISODE_HORIZON}")
print(f"Delta: {DELTA}")
print(f"Agent Epsilon: {AGENT_EPSILON}")
print("\nEnvironment config:")
for key, value in ENV_CONFIG.items():
    print(f"{key}: {value}")

Starting UCBVI experiment with:
Episodes: 500
Horizon: 25
Delta: 0.01
Agent Epsilon: 0.3

Environment config:
map_name: 4x4
is_slippery: True
wind_probabilities: (0.5, 0.125, 0.125, 0.125, 0.125)
render_mode: rgb_array
seed: 42


## Run UCBVI Experiment
Now we'll run the experiment and collect results. This may take a few minutes depending on the number of episodes.

In [3]:
# Run the experiment
cumulative_rewards_history, avg_step_rewards_history, captured_frames = run_ucbvi_experiment(
    num_episodes=NUM_EPISODES,
    env_config=ENV_CONFIG,
    episode_horizon=EPISODE_HORIZON,
    planning_sweeps=PLANNING_SWEEPS,
    delta=DELTA,
    agent_epsilon=AGENT_EPSILON,
    capture_replay=True
)

# Calculate final statistics
final_avg_window = min(100, NUM_EPISODES)
final_avg_reward = np.mean(cumulative_rewards_history[-final_avg_window:])
final_avg_step_reward = np.mean(avg_step_rewards_history[-final_avg_window:])

print(f"\nExperiment Results:")
print(f"Final average cumulative reward (last {final_avg_window} episodes): {final_avg_reward:.2f}")
print(f"Final average reward per step (last {final_avg_window} episodes): {final_avg_step_reward:.3f}")

Initializing environment with config: {'map_name': '4x4', 'is_slippery': True, 'wind_probabilities': (0.5, 0.125, 0.125, 0.125, 0.125), 'render_mode': 'rgb_array', 'seed': 42}
[DEBUG] Initializing Pygame...
Initializing UCBVI agent with: num_states=16, n_actions=4, horizon=25, delta=0.01, epsilon=0.3

--- Episode 1 (UCBVI Diags ON) ---
  UCBVI.plan end (sweeps=25):
    V[0,:]=[3.035 3.035 3.035 3.035]
    V[14,:]=[3.035 3.035 3.035 3.035]
    V[15,:]=[0. 0. 0. 0.]
    Q[14,0,2]=3.035 (s14, intend 0, applied RIGHT to G)
    Q[14,1,2]=3.035 (s14, intend 1, applied RIGHT to G)
    Q[14,2,2]=3.035 (s14, intend 2, applied RIGHT to G)
    Q[14,3,2]=3.035 (s14, intend 3, applied RIGHT to G)
  UCBVI.update: s=0, x_int=3, a_exec=0, r=0.17, s'=4
    N[0,3,0]=1.0->2.0, R[0,3,0]=0.00->0.08
Ep 1, Step 1: s=0, x_int=3, a_app=0, a_exec=0 -> r=0.17, s'=4
  UCBVI.update: s=4, x_int=3, a_exec=2, r=-1.00, s'=5
    N[4,3,2]=1.0->2.0, R[4,3,2]=0.00->-0.50
  UCBVI.update: s=0, x_int=3, a_exec=2, r=0.17, s'=

## Visualize Learning Progress
Let's plot the learning curves showing both cumulative rewards per episode and average rewards per step.

In [4]:
# Create and display the learning curves
plot_title = f"UCBVI on {'Slippery ' if ENV_CONFIG['is_slippery'] else ''}{'Windy ' if ENV_CONFIG['wind_probabilities'] != (1,0,0,0,0) else ''}FrozenLake {ENV_CONFIG['map_name']}"
plot_results(cumulative_rewards_history, avg_step_rewards_history, title=plot_title)

# Display the saved plot
plots_dir = "plots"
plot_filename = plot_title.lower().replace(" ", "_").replace("(", "").replace(")", "").replace(":", "") + ".png"
plot_path = os.path.join(plots_dir, plot_filename)

# Check if plot exists and display it
if os.path.exists(plot_path):
    plt.figure(figsize=(12, 7))
    img = plt.imread(plot_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

Plot saved as plots/ucbvi_on_slippery_windy_frozenlake_4x4.png


## Replay Last Successful Episode
If we captured any successful episodes, let's watch a replay of the last one. This will show how the agent navigates through the environment.

In [5]:
if captured_frames:
    print(f"Replaying successful episode ({len(captured_frames)} frames)")
    replay_episode(captured_frames, fig_title="Last Successful Episode Replay", interval=300)
else:
    print("No successful episodes were captured for replay.")

Replaying successful episode (24 frames)
Preparing replay of 24 frames...
Displaying animation. Close the animation window to continue...
